# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [1]:
%cd sample_data/
!git clone https://github.com/hdmeyer/llm_engineering.git
%cd llm_engineering/
!uv sync
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama
!ollama pull llama3.2:1b

/content/sample_data
Cloning into 'llm_engineering'...
remote: Enumerating objects: 50176, done.
remote: Counting objects: 100% (21764/21764), done.
remote: Compressing objects: 100% (532/532), done.
remote: Total 50176 (delta 21363), reused 21232 (delta 21232), pack-reused 28412 (from 2)
Receiving objects: 100% (50176/50176), 578.28 MiB | 36.10 MiB/s, done.
Resolving deltas: 100% (29172/29172), done.
/content/sample_data/llm_engineering
Using CPython 3.12.12
Creating virtual environment at: .venv
Resolved 267 packages in 1.40s                                       
Prepared 260 packages in 1m 13s                                          
Installed 260 packages in 497ms                             
 + aiofiles==24.1.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.2
 + aiosignal==1.4.0
 + annotated-doc==0.0.3
 + annotated-types==0.7.0
 + ansi2html==1.9.2
 + anthropic==0.72.0
 + anyio==4.11.0
 + asttokens==3.0.0
 + attrs==25.4.0
 + backoff==2.2.1
 + bcrypt==5.0.0
 + beautifulsoup4==4.14.2


In [2]:
!pip install chromadb litellm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 111.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.9/36.9 MB 46.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 165.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━

In [3]:
import subprocess
import sys
print(sys.path[:5])
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from openai import AsyncOpenAI
from dotenv import load_dotenv
#from langchain_huggingface import HuggingFaceEmbeddings
import os
import ollama
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
# Start the command in a background process
process = subprocess.Popen(["ollama", "serve"])

# The kernel can continue execution while the process runs in the background
print("The 'ollama serve' process is running in the background.")


OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
MODEL = 'llama3.2:1b'
#embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
load_dotenv(override=True)


DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "all-MiniLM-L6-v2"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

['/content', '/env/python', '/usr/lib/python313.zip', '/usr/lib/python3.13', '/usr/lib/python3.13/lib-dynload']
The 'ollama serve' process is running in the background.


In [ ]:
##DONT RUN FOR LOCAL

from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)

MODEL = "gpt-4.1-nano"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai = OpenAI()

In [4]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [5]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [6]:
%cd week5/

/content/sample_data/llm_engineering/week5


In [8]:
for folder in KNOWLEDGE_BASE_PATH.iterdir():
    print(folder)

knowledge-base/employees
knowledge-base/company
knowledge-base/contracts
knowledge-base/products


In [9]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [10]:
documents = fetch_documents()

Loaded 76 documents


### Donezo! On to Step 2 - make the chunks

In [11]:
# Semantic chunking
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [12]:
print(documents[0])

{'type': 'employees', 'source': 'knowledge-base/employees/Alex Chen.md', 'text': '# HR Record\n\n# Alex Chen\n\n## Summary\n- **Date of Birth:** March 15, 1990\n- **Job Title:** Backend Software Engineer\n- **Location:** San Francisco, California\n- **Current Salary:** $115,000  \n\n## Insurellm Career Progression\n- **April 2020:** Joined Insurellm as a Junior Backend Developer. Focused on building APIs to enhance customer data security.\n- **October 2021:** Promoted to Backend Software Engineer. Took on leadership for a key project developing a microservices architecture to support the company\'s growing platform.\n- **March 2023:** Awarded the title of Senior Backend Software Engineer due to exemplary performance in scaling backend services, reducing downtime by 30% over six months.\n\n## Annual Performance History\n- **2020:**  \n  - Completed onboarding successfully.  \n  - Met expectations in delivering project milestones.  \n  - Received positive feedback from the team leads.\n\

In [15]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: company
The document has been retrieved from: knowledge-base/company/culture.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 7 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# Insurellm Culture

## Vision Statement
To revolu

In [13]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [14]:
make_messages(documents[0])

[{'role': 'user',
  'content': '\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: employees\nThe document has been retrieved from: knowledge-base/employees/Alex Chen.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don\'t leave anything out.\nThis document should probably be split into 6 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the documen

In [15]:
!ollama pull llama3.2:1b

In [16]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=f"ollama/{MODEL}", messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [20]:
process_document(documents[0])

[Result(page_content="Insurellm Culture Vision Statement\n\nOur mission is to revolutionize the insurance industry through innovative technology that makes insurance accessible, transparent, and effortless for everyone.\n\n# Insurellm Culture\n## Vision Statement\nTo revolutionize the insurance industry through innovative technology that makes insurance accessible, transparent, and effortless for everyone.\n## Mission Statement\nWe empower insurance providers and consumers with cutting-edge software solutions that streamline processes, enhance customer experiences, and drive meaningful connections in the insurance marketplace. By combining deep industry expertise with technological innovation, we're building the future of insurance.\n## Core Values\n### Innovation First\nWe challenge the status quo and embrace creative problem-solving. Our team is encouraged to experiment, take calculated risks, and push the boundaries of what's possible in insurance technology. We believe that breakth

In [17]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [21]:
for doc in documents[:5]:
    print(doc)

{'type': 'employees', 'source': 'knowledge-base/employees/Alex Chen.md', 'text': '# HR Record\n\n# Alex Chen\n\n## Summary\n- **Date of Birth:** March 15, 1990\n- **Job Title:** Backend Software Engineer\n- **Location:** San Francisco, California\n- **Current Salary:** $115,000  \n\n## Insurellm Career Progression\n- **April 2020:** Joined Insurellm as a Junior Backend Developer. Focused on building APIs to enhance customer data security.\n- **October 2021:** Promoted to Backend Software Engineer. Took on leadership for a key project developing a microservices architecture to support the company\'s growing platform.\n- **March 2023:** Awarded the title of Senior Backend Software Engineer due to exemplary performance in scaling backend services, reducing downtime by 30% over six months.\n\n## Annual Performance History\n- **2020:**  \n  - Completed onboarding successfully.  \n  - Met expectations in delivering project milestones.  \n  - Received positive feedback from the team leads.\n\

In [22]:
## Normally you would do for all, but my gpu cries when doing all
#chunks = create_chunks(documents)
chunks = create_chunks(documents[:5])

100%|██████████| 5/5 [09:16<00:00, 111.26s/it]


In [23]:
print(len(chunks))

14


In [27]:
print(chunks[5])

page_content='# Insurellm Career Progression\n\nInsurellm Career Progression - **2020** - Joined Insurellm as a HR Coordinator, Responsibilities included assisting with recruitment processes and managing employee onboarding.\n- **2021** - Promoted to HR Generalist, Transitioned to a role with expanded responsibilities, including handling employee relations and benefits administration.\n- **2022** - Completed the HR Leadership Development Program, Enhanced skills in conflict resolution and strategic planning.\n- **2023** - Actively involved in initiating the company’s Diversity and Inclusion programs, Samantha Greene played a key role in launching mentorship initiatives and employee resource groups.\n\n[Annual Performance History]\n- **2020:** Exceeds Expectations  Samantha Greene demonstrated exceptional organizational skills and contributed to a streamlined onboarding process, earning commendations from senior leadership.\n- **2021:** Meets Expectations  While proficient in her new ro

### Well that was easy! If a bit slow.

In the python module version, I sneakily use the multi-processing Pool to run this in parallel,
but if you get a Rate Limit Error you can turn this off in the code.

### Finally, Step 3 - save the embeddings

In [33]:
!pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 18.9 MB/s eta 0:00:0000:01


In [34]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]

    #emb = openai.embeddings.create(model=embedding_model, input=texts).data
    #vectors = [e.embedding for e in emb]
    # OLLAMA
    vectors = model.encode(
        texts,
        convert_to_numpy=True
    ).tolist()
    # OLLAMA end
    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [35]:
create_embeddings(chunks)

Vectorstore created with 14 documents


# Nothing more to do here... right?

Wait! Didja think I'd forget??

In [36]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [39]:
tsne = TSNE(n_components=2, random_state=42, perplexity=5) ## for my example this is only 5, but should be 76 or so.
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

#fig.show()
fig.write_html("chart3.html")

In [40]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [45]:
!cp chart4.html /content/drive/MyDrive/

cp: cannot create regular file '/content/drive/MyDrive/': No such file or directory


In [42]:
tsne = TSNE(n_components=3, random_state=42, perplexity=5) ## for my example this is only 5, but should be 76 or so.
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

#fig.show()
fig.write_html("chart4.html")

In [44]:
from google.colab import drive

drive.flush_and_unmount()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [46]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [54]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(f"ollama/{MODEL}", messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [50]:
RETRIEVAL_K = 10
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
def fetch_context_unranked(question):
    #query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    # OLLAMA
    query = model.encode(
            [question],
            convert_to_numpy=True
        ).tolist()
        # OLLAMA end
    results = collection.query(query_embeddings=query, n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [73]:
#question = "Who won the IIOTY award?"
question = "Who has strong design leadership?"
chunks = fetch_context_unranked(question)

In [74]:
for chunk in chunks:
    print(chunk.page_content[:30]+"...")

Michelle Rivera: Design System...
Michelle Rivera: Senior UX/UI ...
Michelle Rivera: Project Manag...
Michelle Rivera: Product Desig...
Michelle Rivera: UX Designer

...
Michelle Rivera: Digital Produ...
# Samantha Greene

Summary - *...
# Insurellm Career Progression...
# HR Record

Overview of Maya ...
# Compensation History

Compen...
Lisa Anderson

# Summary

- **...
# Other HR Notes

Other HR Not...
HR Record

# Lisa Anderson

# ...
Insurellm Career Progression

...


In [75]:
reranked = rerank(question, chunks)

[8, 9, 10, 11, 12, 13, 14]


In [76]:
for chunk in reranked:
    print(chunk.page_content[:50]+"...")

# Insurellm Career Progression

Insurellm Career P...
# HR Record

Overview of Maya Thompson's HR record...
# Compensation History

Compensation History - **2...
Lisa Anderson

# Summary

- **Date of Birth:** Nov...
# Other HR Notes

Other HR Notes - Samantha Greene...
HR Record

# Lisa Anderson

# HR Record

# HR Reco...
Insurellm Career Progression

# Insurellm Career P...


In [77]:
question = "Who has strong design leadership?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "HR" in c.page_content.lower():
        print(index)

In [78]:
reranked = rerank(question, chunks)

[8, 9, 10, 11, 12, 13, 14, 1, 2, 3, 4, 5, 6]


In [80]:
for index, c in enumerate(reranked):
    if "design leadership" in c.page_content.lower():
        print(index)

7
8
9
10
11
12


In [81]:
reranked[0].page_content

'# Insurellm Career Progression\n\nInsurellm Career Progression - **2020** - Joined Insurellm as a HR Coordinator, Responsibilities included assisting with recruitment processes and managing employee onboarding.\n- **2021** - Promoted to HR Generalist, Transitioned to a role with expanded responsibilities, including handling employee relations and benefits administration.\n- **2022** - Completed the HR Leadership Development Program, Enhanced skills in conflict resolution and strategic planning.\n- **2023** - Actively involved in initiating the company’s Diversity and Inclusion programs, Samantha Greene played a key role in launching mentorship initiatives and employee resource groups.\n\n[Annual Performance History]\n- **2020:** Exceeds Expectations  Samantha Greene demonstrated exceptional organizational skills and contributed to a streamlined onboarding process, earning commendations from senior leadership.\n- **2021:** Meets Expectations  While proficient in her new role, Samantha 

In [82]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [83]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [84]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [85]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=f"ollama/{MODEL}", messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [86]:
rewrite_query("Who has strong design leadership?", [])

"Can you find information on the company's design leadership team?"

In [87]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=f"ollama/{MODEL}", messages=messages)
    return response.choices[0].message.content, chunks

In [88]:
answer_question("Who has strong design leadership?", [])

"What are some notable design leaders in the insurance industry known for their innovative approach to customer experience?"
[8, 9, 10, 11, 14]


('According to the user\'s question, the answer is Lisa Anderson. Her HR Record highlights that she "Manages team of 5 marketing specialists" and "Known for developing talent and creating high-performing culture", indicating that she has strong leadership skills in design and talent management.',
 [Result(page_content="# Other HR Notes\n\nOther HR Notes - Samantha Greene has expressed interest in pursuing an HR certification (SHRM-CP) to further her career growth within Insurellm.\n- Participated in Insurellm's employee wellness program, promoting mental health resources among staff.\n- Actively volunteers with local nonprofits and encourages staff involvement in community outreach programs, enhancing Insurellm's corporate social responsibility initiatives.\n\n[Other HR Notes]", metadata={'source': 'knowledge-base/employees/Samantha Greene.md', 'type': 'employees'}),
  Result(page_content='# Samantha Greene\n\nSummary - **Date of Birth:** October 14, 1990, **Job Title:** HR Generalist,

In [ ]:
#answer_question("Who went to Manchester University?", [])

In [91]:
def answer_question_without_qr(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    #query = rewrite_query(question, history)
    print(question)
    chunks = fetch_context(question)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=f"ollama/{MODEL}", messages=messages)
    return response.choices[0].message.content, chunks
print("Without query rewriting?")
answer_question_without_qr("Who has strong design leadership?", [])

Without query rewriting?
Who has strong design leadership?
[]


("At Insurellm, we believe that strong design leadership is essential for creating innovative and user-friendly digital products.\n\nWhile there isn't a specific title that represents strong design leadership, we can look at various roles that require strong design skills and leadership qualities. Some of these roles might include:\n\n* Senior Designer: Overseeing the design team and ensuring that designs meet the company's standards and client expectations.\n* UX/UI Designer: Leading the design process to create user-centered and visually appealing experiences for clients.\n* Design Director: Providing overall design direction and overseeing the design team to deliver high-quality designs.\n\nIt's worth noting that Insurellm is a digital insurance platform, and design leadership might not be a direct role within our company. However, our team members are skilled designers who work together to create an exceptional user experience for our clients.\n\nIf you have any specific questions 